# 06 — CTU-UHB Full Feature Extraction

Applies the validated ASTV/ALTV methodology from `05_ctu_feasibility_test.ipynb`
across all CTU-UHB records to build a full feature dataset for external
validation (evaluation to be run by Shreya).

**v3 update:** v2 (signal cleaning: physiological range clip, jump-artifact
removal, short-gap interpolation) improved ASTV/MSTV agreement with UCI but
left Variance, Width, Min/Max, and MLTV substantially distorted (see
comparison table history below). Diagnosis: those artifact checks target
*noise*, but the residual distortion is more consistent with genuine
acceleration/deceleration episodes being included in features that SisPorto's
underlying Porto-system methodology computes over the FHR **baseline** only
(Bernardes et al. 1996, "A more objective fetal heart rate baseline
estimation," BJOG 103:714-715 — baseline defined during stable segments, in
the absence of accelerations/decelerations/contractions. Note: cited via
Bernardes et al. 2022's retrospective review, "Computerized analysis of
cardiotocograms... thirty-two years after" (J Perinat Med), which attributes
this exclusion principle to the 1996 paper specifically -- the 1996 primary
source itself is paywalled and has not been independently read in full;
this is a secondhand citation, flagged as such rather than presented as
directly verified. An earlier version of this notebook incorrectly cited
Bernardes et al. 1991 for this point -- checked and corrected, since 1991
only describes baseline estimation as an unresolved "challenge" without
detailing the exclusion methodology itself.) This version excludes detected
acceleration and deceleration episodes before computing histogram-based
features (Width, Min, Max, Mode, Mean, Median, Variance), baseline (LB), and
the long-term-variability features (MLTV, ALTV) — since a window's range is
also inflated by an included accel/decel swing rather than genuine
beat-to-beat variability — consistent with that description.

**Note:** CTU-UHB provides only FHR and UC signal channels. Some UCI features
(e.g. FM — fetal movement) have no direct signal source and are not
implemented — flagged in Known Limitations below.

**Citation update (Interim Report phase):** the ASTV/ALTV rules implemented below are now attributed to a specific source that names SisPorto directly — Costa M, Xavier M, Nunes I, Henriques TS, "Fetal Heart Rate Fragmentation," Front Pediatr. 2021;9:662101 — rather than general literature. This paper also ran the real SisPorto software on this same CTU-UHB database and reports its actual output (median abnormal STV 44-46%, median abnormal LTV ~2%), giving a same-dataset benchmark in addition to the UCI-distribution comparison used throughout this notebook. See the second comparison table below.

In [10]:
import os
import wfdb
import numpy as np
import pandas as pd

folder = '../data/external/ctu-chb-intrapartum-cardiotocography-database-1.0.0'
hea_files = sorted([f for f in os.listdir(folder) if f.endswith('.hea')])
record_ids = [f.replace('.hea', '') for f in hea_files]
print(f"Total records found: {len(record_ids)}")

Total records found: 552


## Signal cleaning

Raw FHR channels contain dropout artifacts (values far outside physiological
range) and sudden implausible jumps between adjacent samples. `clean_fhr_signal()`:

1. Marks samples outside the physiological range (50-200bpm) as missing.
2. Marks samples that jump by more than 25bpm from the previous sample as
   artifacts and marks them missing too.
3. Linearly interpolates gaps up to 15 seconds; longer gaps are left as
   missing and dropped before feature extraction.

In [11]:
def clean_fhr_signal(fhr, fs, low=50, high=200, max_jump=25, gap_limit_sec=15):
    """Clip to physiological range, remove sudden-jump artifacts, interpolate
    short gaps. Returns a cleaned array (may still contain NaN for long gaps).
    """
    sig = fhr.astype(float).copy()
    sig[(sig < low) | (sig > high)] = np.nan

    diffs = np.abs(np.diff(sig))
    jump_idx = np.where(diffs > max_jump)[0] + 1
    sig[jump_idx] = np.nan

    gap_limit = max(1, int(gap_limit_sec * fs))
    s = pd.Series(sig).interpolate(limit=gap_limit, limit_direction='both')
    return s.to_numpy()


def get_valid(fhr_clean):
    """Drop any remaining NaN (long dropout gaps) before feature extraction."""
    return fhr_clean[~np.isnan(fhr_clean)]

## Baseline segment isolation

Bernardes et al. (1996), "A more objective fetal heart rate baseline
estimation" (BJOG 103:714-715) — cited via Bernardes et al. (2022)'s
retrospective review, since the 1996 primary paper is paywalled and not
independently verified in full text — describes the FHR baseline as
computed "during fetal rest... in the absence of fetal movements, uterine
contractions, drug actions or other abnormal stimuli." SisPorto's
histogram-based features (Width, Min, Max, Mode, Mean, Median, Variance)
and LB are baseline descriptors, so they should be computed on this same
baseline-only segment, not the full trace.

**Correction note:** an earlier version of this notebook cited Bernardes et
al. (1991) for this specific exclusion principle. Checked directly against
a 2022 retrospective review by the same lead author: the 1991 paper is only
referenced there as establishing baseline estimation as an early *unsolved
challenge*, with no exclusion methodology detailed. The exclusion principle
itself is attributed by that review to the 1996 paper (and a 2004
follow-up), not 1991. Citation corrected accordingly.

**Still unverified:** no source checked so far -- not Bernardes 1991/1996,
not the 2022 review, not Costa et al. (2021) -- documents SisPorto's actual
histogram-construction algorithm (bin width, exact statistical treatment,
etc.). The baseline-exclusion *principle* is reasonably well-supported; the
specific *histogram calculation* implemented below remains an unverified
reconstruction, not a documented replication. Flagged here explicitly
rather than left implicit.

`get_ac_dc_mask()` flags any sample that is part of a sustained rise or fall
of >=15bpm lasting >=15s (the same accel/decel definition already used for
the AC feature, matching FIGO's clinical definition of an acceleration --
Ayres-de-Campos, Spong, Chandraharan, "FIGO consensus guidelines on
intrapartum fetal monitoring," Int J Gynecol Obstet 2015;131(1):13-24), so
those episodes can be excluded before the histogram is built.

**Known simplification:** the baseline used to detect accel/decel episodes
here is a single global median across the whole recording. FIGO's actual
definition uses a local, time-varying baseline (e.g. a 10-minute rolling
window), not one fixed value for the entire trace. This is a simplification
made for implementation speed, not a documented equivalence -- flagged as
a limitation below.

In [12]:
def get_ac_dc_mask(fhr_valid, fs, threshold_bpm=15, min_dur_sec=15):
    """Boolean mask, True where a sample belongs to a sustained
    acceleration or deceleration episode (>=15bpm from baseline, >=15s).
    """
    baseline_est = np.median(fhr_valid)
    min_samples = int(min_dur_sec * fs)
    mask = np.zeros(len(fhr_valid), dtype=bool)

    for direction in (1, -1):
        above = direction * (fhr_valid - baseline_est) > threshold_bpm
        run_start = None
        for i, val in enumerate(above):
            if val and run_start is None:
                run_start = i
            elif not val and run_start is not None:
                if i - run_start >= min_samples:
                    mask[run_start:i] = True
                run_start = None
        if run_start is not None and len(above) - run_start >= min_samples:
            mask[run_start:] = True
    return mask

In [13]:
def compute_astv_final(fhr_valid, fs, threshold_bpm=1.0, lag_sec=1.0):
    """Validated in 05: mean=44.6 vs UCI mean=47 (n=20 sample test).
    Corroborated by Costa et al. (2021, Front Pediatr 9:662101), which states
    SisPorto's rule explicitly: "percentage of subsequent FHR signals
    differing less than 1bpm" — and reports real SisPorto output on this
    same CTU-UHB dataset (median abnormal STV 44-46%)."""
    lag = int(lag_sec * fs)
    if len(fhr_valid) <= lag:
        return None
    diffs = np.abs(fhr_valid[lag:] - fhr_valid[:-lag])
    return 100 * (diffs < threshold_bpm).mean()

def compute_mstv(fhr_valid, fs, window_sec=60):
    """Mean value of short-term variability (bpm)."""
    window_size = int(window_sec * fs)
    diffs = np.abs(np.diff(fhr_valid))
    windows = [diffs[i:i+window_size].mean() for i in range(0, len(diffs), window_size) if len(diffs[i:i+window_size]) > 0]
    return np.mean(windows) if windows else None

def compute_altv_v2(fhr_valid, fs, window_sec=60, range_threshold_bpm=5.0):
    """Percentage of 60s windows with range <= 5bpm (abnormal LTV).
    Corroborated by Costa et al. (2021): SisPorto's LTV rule is "percentage
    of FHR signals with a difference between the minimum and maximum values
    in a 1 min window lower than 5bpm" — real SisPorto output on this same
    CTU-UHB dataset gives median abnormal LTV ~2%."""
    window_size = int(window_sec * fs)
    abnormal, total = 0, 0
    for i in range(0, len(fhr_valid) - window_size, window_size):
        seg = fhr_valid[i:i+window_size]
        if len(seg) == 0:
            continue
        abnormal += (seg.max() - seg.min() <= range_threshold_bpm)
        total += 1
    return 100 * abnormal / total if total else None

def compute_mltv(fhr_valid, fs, window_sec=60):
    """Mean value of long-term variability: average window range."""
    window_size = int(window_sec * fs)
    ranges = []
    for i in range(0, len(fhr_valid) - window_size, window_size):
        seg = fhr_valid[i:i+window_size]
        if len(seg) == 0:
            continue
        ranges.append(seg.max() - seg.min())
    return np.mean(ranges) if ranges else None

def compute_ac(fhr_valid, fs, rise_bpm=15, min_dur_sec=15):
    """Count of rises >=15bpm above baseline lasting >=15s."""
    baseline = np.median(fhr_valid)
    threshold = baseline + rise_bpm
    above = fhr_valid > threshold
    min_samples = int(min_dur_sec * fs)
    count, run_len = 0, 0
    for val in above:
        if val:
            run_len += 1
        else:
            if run_len >= min_samples:
                count += 1
            run_len = 0
    if run_len >= min_samples:
        count += 1
    return count

def compute_histogram_features(fhr_baseline_only):
    """Width, Min, Max, Mode, Mean, Median, Variance from the FHR baseline
    distribution (accelerations/decelerations excluded)."""
    n_bins = max(1, int(fhr_baseline_only.max() - fhr_baseline_only.min()))
    hist, bin_edges = np.histogram(fhr_baseline_only, bins=n_bins)
    mode_val = bin_edges[np.argmax(hist)]
    return {
        'Width': fhr_baseline_only.max() - fhr_baseline_only.min(),
        'Min': fhr_baseline_only.min(),
        'Max': fhr_baseline_only.max(),
        'Mode': mode_val,
        'Mean': fhr_baseline_only.mean(),
        'Median': np.median(fhr_baseline_only),
        'Variance': fhr_baseline_only.var(),
    }

In [14]:
results = []
errors = []

MIN_BASELINE_FRACTION = 0.3  # if AC/DC exclusion leaves too little signal, skip baseline features

for rid in record_ids:
    try:
        rec = wfdb.rdrecord(os.path.join(folder, rid))
        fhr_raw = rec.p_signal[:, 0]
        fs = rec.fs

        fhr_clean = clean_fhr_signal(fhr_raw, fs)
        fhr_valid = get_valid(fhr_clean)

        if len(fhr_valid) < fs * 60:
            errors.append((rid, "insufficient valid signal after cleaning"))
            continue

        ac_dc_mask = get_ac_dc_mask(fhr_valid, fs)
        fhr_baseline = fhr_valid[~ac_dc_mask]

        if len(fhr_baseline) < MIN_BASELINE_FRACTION * len(fhr_valid):
            fhr_baseline = fhr_valid  # fall back rather than compute on a tiny, unrepresentative slice

        row = {'record_id': rid}
        row['ASTV'] = compute_astv_final(fhr_valid, fs)
        row['MSTV'] = compute_mstv(fhr_valid, fs)
        row['ALTV'] = compute_altv_v2(fhr_baseline, fs)
        row['MLTV'] = compute_mltv(fhr_baseline, fs)
        row['AC'] = compute_ac(fhr_valid, fs)
        row.update(compute_histogram_features(fhr_baseline))
        row['LB'] = np.median(fhr_baseline)
        results.append(row)
    except Exception as e:
        errors.append((rid, str(e)))

print(f"Extracted: {len(results)} | Failed: {len(errors)}")
if errors:
    print("First few errors:", errors[:5])

Extracted: 552 | Failed: 0


In [15]:
features_df = pd.DataFrame(results)
os.makedirs('../data/processed', exist_ok=True)
features_df.to_csv('../data/processed/ctu_extracted_features.csv', index=False)
features_df.describe()

,ASTV,MSTV,ALTV,MLTV,AC,Width,Min,Max,Mode,Mean,Median,Variance,LB
count,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000,552.000000
mean,42.714248,0.676465,1.508827,28.316402,6.461957,99.462706,78.913200,178.375906,139.116403,137.499046,138.270225,111.674073,138.270225
std,10.837161,0.205020,3.817824,8.249372,5.455180,23.089643,17.164312,14.834729,12.925881,12.126101,12.342896,56.636841,12.342896
min,12.600510,0.322876,0.000000,6.812500,0.000000,34.163636,50.000000,141.500000,100.336283,100.783546,102.000000,20.850492,102.000000
25%,36.234964,0.542342,0.000000,22.633030,2.000000,83.000000,66.750000,167.187500,131.147671,129.203265,129.958333,78.687887,129.958333
50%,43.700749,0.617519,0.000000,27.443793,5.000000,98.500000,77.750000,176.500000,138.893061,136.719190,137.500000,101.954025,137.500000
75%,49.510721,0.763801,1.785714,33.020392,10.000000,115.250000,90.250000,192.812500,147.000000,145.358884,146.500000,135.098291,146.500000
max,79.632055,1.466125,52.083333,66.152612,28.000000,149.500000,146.500000,200.000000,186.500000,180.211165,183.500000,781.505387,183.500000


## Comparison against UCI reference distribution

Sanity check: cleaned, baseline-isolated CTU-UHB feature distributions should
sit in the same ballpark as the UCI dataset used to train the model (same
underlying clinical signal, different acquisition/preprocessing pipeline —
exact match is not expected, but order-of-magnitude agreement is).

In [16]:
uci = pd.read_csv('../data/processed/cleaned_ctg_data.csv')
cols = ['ASTV', 'MSTV', 'ALTV', 'MLTV', 'AC', 'Width', 'Min', 'Max', 'Mode', 'Mean', 'Median', 'Variance', 'LB']

comparison = pd.DataFrame({
    'UCI_mean': uci[cols].mean(),
    'CTU_mean': features_df[cols].mean(),
    'UCI_std': uci[cols].std(),
    'CTU_std': features_df[cols].std(),
    'UCI_min': uci[cols].min(),
    'CTU_min': features_df[cols].min(),
    'UCI_max': uci[cols].max(),
    'CTU_max': features_df[cols].max(),
})
comparison['mean_ratio_CTU_over_UCI'] = comparison['CTU_mean'] / comparison['UCI_mean']
comparison

,UCI_mean,CTU_mean,UCI_std,CTU_std,UCI_min,CTU_min,UCI_max,CTU_max,mean_ratio_CTU_over_UCI
ASTV,46.978693,42.714248,17.167716,10.837161,12.0,12.600510,87.0,79.632055,0.909226
MSTV,1.335559,0.676465,0.884232,0.205020,0.2,0.322876,7.0,1.466125,0.506504
ALTV,9.759943,1.508827,18.270136,3.817824,0.0,0.000000,91.0,52.083333,0.154594
MLTV,8.169176,28.316402,5.633034,8.249372,0.0,6.812500,50.7,66.152612,3.466249
AC,2.733902,6.461957,3.567741,5.455180,0.0,0.000000,26.0,28.000000,2.363639
Width,70.566288,99.462706,38.990851,23.089643,3.0,34.163636,180.0,149.500000,1.409493
Min,93.539299,78.913200,29.546379,17.164312,50.0,50.000000,159.0,146.500000,0.843637
Max,164.105587,178.375906,17.947492,14.834729,122.0,141.500000,238.0,200.000000,1.086958
Mode,137.448390,139.116403,16.403636,12.925881,60.0,100.336283,187.0,186.500000,1.012136
Mean,134.592803,137.499046,15.610971,12.126101,73.0,100.783546,182.0,180.211165,1.021593


## Comparison against real SisPorto output on CTU-UHB itself

The UCI comparison above checks order-of-magnitude plausibility against a
*different* dataset. A stronger check: Costa et al. (2021) ran the actual
SisPorto software on this same CTU-UHB database (last-hour segments, n=246
after excluding high signal-loss records) and reported its real output. This
lets ASTV, ALTV, and LB be checked against genuine SisPorto values on the
correct dataset, not just a same-ballpark UCI comparison.

In [17]:
# Real SisPorto output on CTU-UHB, from Costa et al. (2021), Table 2
# (medians across Normal + Pathologic groups, combined for a single reference range)
real_sisporto_ctu = {
    'LB':   {'median_low': 128, 'median_high': 136},
    'ASTV': {'median_low': 44,  'median_high': 46},
    'ALTV': {'median_low': 2,   'median_high': 2},
}

for feat, ref in real_sisporto_ctu.items():
    our_mean = features_df[feat].mean()
    our_median = features_df[feat].median()
    print(f"{feat}: our mean={our_mean:.1f}, our median={our_median:.1f} "
          f"| real SisPorto-on-CTU-UHB median range={ref['median_low']}-{ref['median_high']}")

LB: our mean=138.3, our median=137.5 | real SisPorto-on-CTU-UHB median range=128-136
ASTV: our mean=42.7, our median=43.7 | real SisPorto-on-CTU-UHB median range=44-46
ALTV: our mean=1.5, our median=0.0 | real SisPorto-on-CTU-UHB median range=2-2


## Ground-truth outcome labels

CTU-UHB has no expert NSP annotation like UCI. Every record's `.hea` header
does include real clinical outcome measures: umbilical artery pH, base
deficit (BDecf), and Apgar scores. These are the standard clinical proxies
used in perinatal research for intrapartum fetal distress when expert
NSP-style labels aren't available.

**Pathological threshold: pH < 7.00 AND BDecf >= 12mmol/L.** This is now
corroborated by three independent sources, not one:
1. MacLennan A, "A template for defining a causal relation between acute
   intrapartum events and cerebral palsy: international consensus
   statement," BMJ. 1999;319:1054-1059 -- the original consensus criterion.
2. ACOG Committee Opinion No. 348 (2006), "Umbilical cord blood gas and
   acid-base analysis" -- adopts the same pH<7.00, BD>=12mmol/L cutoff for
   severe acidemia.
3. FIGO (2015) intrapartum monitoring standards -- defines metabolic
   acidemia as pH<7.0 and/or BE<=-12mmol/L, consistent with the same
   threshold.
This is a well-anchored, multiply-corroborated cutoff.

**Normal threshold: pH >= 7.20.** Checked directly rather than assumed --
this is a commonly used research convention for "any degree of acidemia"
(seen across multiple studies), but unlike the Pathological threshold above,
no single official body (ACOG/FIGO) was found pinning this exact number as
*the* standard normal/abnormal boundary -- ACOG/AAP's own clinical threshold
is the pH<7.00 severe cutoff, not 7.20. Flagged honestly: this Normal/Suspect
boundary is a reasonable, literature-consistent choice, not a
definitively-anchored one the way the Pathological threshold is.

**Suspect**: everything in between (pH < 7.20 but not meeting the
Pathological criterion above) -- analogous to the UCI Suspect class sitting
between clearly-normal and clearly-pathological.

11 records are missing the BDecf field; these fall back to a pH-only
classification (Suspect if pH < 7.20, since a low pH is still noteworthy
even without base-deficit confirmation, but not labelled Pathological
without meeting the full two-part criterion).

Costa et al. (2021)'s own single cutoff (pH <= 7.15) sits inside the
Suspect band here, so this isn't a contradiction of that source -- it's a
finer-grained version consistent with it.

In [18]:
import re

def extract_outcome(rid, folder):
    """Parse pH, BDecf (base deficit), Apgar1, Apgar5 from a CTU-UHB
    record's .hea header."""
    with open(os.path.join(folder, f'{rid}.hea')) as f:
        content = f.read()
    out = {}
    for field in ['pH', 'BDecf', 'Apgar1', 'Apgar5']:
        m = re.search(rf'#{field}\s+(-?[\d.]+)', content)
        out[field] = float(m.group(1)) if m else None
    return out

outcomes = []
for rid in features_df['record_id']:
    row = {'record_id': rid}
    row.update(extract_outcome(rid, folder))
    outcomes.append(row)

outcomes_df = pd.DataFrame(outcomes)
print(f"Records with pH available: {outcomes_df['pH'].notna().sum()} / {len(outcomes_df)}")
print(f"Records with BDecf available: {outcomes_df['BDecf'].notna().sum()} / {len(outcomes_df)}")

def classify_ctu(row):
    """3-class label matching the model's Normal/Suspect/Pathological scheme.
    Normal: pH >= 7.20 (standard acidemia threshold).
    Pathological: pH < 7.00 AND BDecf >= 12 (MacLennan et al. 1999 consensus
    criterion for clinically significant intrapartum metabolic acidosis).
    Suspect: everything else. Missing BDecf falls back to pH-only (Suspect,
    not Pathological, since the full two-part criterion can't be confirmed).
    """
    ph = row['pH']
    bdecf = row['BDecf']
    if pd.isna(ph):
        return None
    if ph < 7.00 and pd.notna(bdecf) and bdecf >= 12:
        return 'Pathological'
    elif ph < 7.20:
        return 'Suspect'
    else:
        return 'Normal'

outcomes_df['CTU_label'] = outcomes_df.apply(classify_ctu, axis=1)
print(outcomes_df['CTU_label'].value_counts())

features_df = features_df.merge(outcomes_df, on='record_id')
features_df.to_csv('../data/processed/ctu_extracted_features.csv', index=False)
print(f"\nSaved {len(features_df)} records with features + 3-class outcome labels to ctu_extracted_features.csv")

Records with pH available: 552 / 552
Records with BDecf available: 541 / 552
CTU_label
Normal          375
Suspect         164
Pathological     13
Name: count, dtype: int64

Saved 552 records with features + 3-class outcome labels to ctu_extracted_features.csv


## Methodology iteration history (for report transparency)

| Version | Change | Result |
|---|---|---|
| v1 | `fhr > 50` cutoff only, no cleaning | Mean/Median/Mode/LB/ASTV/AC close to UCI; Variance ~17x too high, Width ~2x, MLTV ~4.4x too high, ALTV near-zero |
| v2 | + `clean_fhr_signal()`: physiological range clip, jump-artifact removal, gap interpolation | ASTV/MSTV improved; Variance/Width/MLTV/ALTV barely changed (still ~18x/1.7x/4.2x/near-zero) -- confirmed distortion was not from point-artifacts |
| v3 | + baseline-only isolation for histogram features (exclude AC/DC episodes) | Variance 18.1x -> 5.9x, Width 1.7x -> 1.4x, MLTV 4.2x -> 3.5x too high; Min/Mode/Mean/Median/LB all within ~2-15% of UCI |
| Citation update | ASTV/ALTV rules corroborated by Costa et al. (2021); baseline-exclusion citation corrected from Bernardes 1991 to Bernardes 1996 after direct verification | ASTV (42.7 vs real median 44-46) and ALTV (1.5 vs real median ~2) validated against real SisPorto output on CTU-UHB itself |
| Label build | 3-class Normal/Suspect/Pathological label from pH/BDecf, cross-checked against 3 independent clinical sources for the Pathological threshold | 375 Normal / 164 Suspect / 13 Pathological |

## Known Limitations

- **FM (fetal movement), DL/DS/DP (deceleration sub-type counts), Nmax,
  Nzeros, Tendency**: not implemented -- require deeper UC-channel/signal
  morphology analysis beyond this project's scope to date.
- **Primary SisPorto methodology paper** (Ayres-de-Campos et al., 2000, *J
  Matern Fetal Med* 9(5):311-8) remains inaccessible -- requested via
  library/author, not received. In its absence, this notebook draws on
  multiple independently-verified corroborating sources rather than one
  primary paper:
  1. **ASTV/ALTV rules** -- Costa M, Xavier M, Nunes I, Henriques TS,
     "Fetal Heart Rate Fragmentation," Front Pediatr. 2021;9:662101. Read
     in full; states SisPorto's rule explicitly and reports real SisPorto
     output on this same CTU-UHB dataset. The strongest-supported part of
     this methodology.
  2. **Baseline-exclusion principle for histogram features** -- Bernardes
     et al. (1996), BJOG 103:714-715, cited via a 2022 retrospective review
     by the same lead author (checked directly). Note: the 1996 primary
     paper itself is paywalled and has not been independently read in full
     -- this is a secondhand citation, disclosed as such. An earlier
     version of this notebook mis-cited Bernardes et al. (1991) for this
     point; corrected after direct verification showed 1991 does not
     contain this detail.
  3. **Histogram-construction algorithm itself** (bin width, exact
     statistical treatment for Width/Min/Max/Mode/Variance) -- **not
     documented in any source checked so far.** This remains an unverified
     reconstruction. Explicitly the weakest-supported part of the
     methodology; flagged rather than implied to be equivalent to (1) or
     (2) above.
- **Accel/decel baseline detection uses a single global median** across
  each whole recording as the reference point, not a local/time-varying
  baseline. FIGO's actual definition uses a rolling local baseline (e.g. a
  10-minute window). This is an implementation simplification, not a
  documented equivalence -- flagged as a limitation, not hidden.
- **Ground-truth labels**: 3-class label built from pH/BDecf. The
  Pathological threshold (pH<7.00 AND BDecf>=12) is corroborated by three
  independent sources (MacLennan et al. 1999; ACOG Committee Opinion 348,
  2006; FIGO 2015) -- well-anchored. The Normal threshold (pH>=7.20) is a
  common research convention, checked directly, but no single official
  body was found pinning this exact number as *the* standard -- disclosed
  as the weaker-anchored of the two cutoffs, not presented as equally solid.
  This is also a clinical-outcome proxy, not an expert CTG-pattern
  annotation like UCI's NSP label -- measures a related but not identical
  thing.
- **Signal cleaning thresholds** (50-200bpm range, 25bpm jump limit, 15s
  gap interpolation) are standard CTG clinical plausibility bounds rather
  than SisPorto-internal constants, since the latter are not published.